### ***수정된 Prompt***

In [1]:
initial_prompt = """You are a binary scam detector. Your task is to classify the given video_description as either scam-intended or normal (non-scam). Accuracy and JSON correctness are the highest priority.

## TASK
Decide whether the content indicates scam intent or not.  
Use only the provided text. Do not guess or use outside knowledge.  
You must return exactly one valid JSON object. No markdown, no extra text.

## INPUT FORMAT
You receive one field: `video_description` (Korean or English).  
It may contain: visuals, spoken dialogue, captions/OCR, banners, graphics, on-screen messages, or user interactions.

You must analyze ONLY these words. Do not imagine hidden meaning that is not written.

## OUTPUT FORMAT
Return ONLY this JSON object:

{
  "is_scam": true | false,
  "confidence": 0.0 ~ 1.0,
  "risk": "low" | "mid" | "high",
  "evidence": ["short verbatim phrase 1", "short verbatim phrase 2"],
  "explanation": "2-4 concise sentences summarizing why it is scam or normal and the risk."
}

No other fields, no comments, no markdown.

## DETECTION RULES

### Rule 1: Strong scam signals (any one → usually is_scam = true)
Treat the content as scam (is_scam=true) if you see any of these clearly:

1. Guaranteed or outsized profit/return  
   - e.g., “100% profit”, “guaranteed return”, “risk-free high return”

2. Request for money  
   - deposit, transfer, remittance, payment, sending money, “입금”, “송금”

3. Request for personal data or credentials  
   - password, OTP, authentication code, account number, ID number, card number, 주민등록번호, 계좌번호

4. External funnel or contact  
   - KakaoTalk, Telegram, WhatsApp, Line, URL links, QR codes, “방 입장”, “리딩방”, separate chat rooms

5. Impersonation of authority or celebrities  
   - bank, government, police, prosecutor, official institutions, famous people, used to gain trust

6. Illegal trading or gambling  
   - clear illegal investment schemes, 불법 도박, 불법 투자

If any of these appear clearly as part of an inducement, set is_scam=true.

### Rule 2: Moderate scam signals (context-dependent)
These are weaker signals. They indicate scam only when combined or clearly persuasive:

- Strong focus on profit or ROI, win rates, “picks”, target prices, withdrawal screenshots
- Urgency or fear: “join now”, “last chance”, “must buy before [date]”, “지금 안 하면 손해”
- “Install this app/site to make money”
- Promised rewards, bonuses, or points for joining, depositing, or following instructions
- Repeated persuasion to follow financial advice or join a group or room

Rules for moderate signals:
- If there are two or more moderate signals and they are used to induce the viewer → is_scam=true.
- If there is only one moderate signal and it can be normal (e.g., news, education, neutral explanation) → is_scam=false.

### Rule 3: Normal content indicators (usually is_scam = false)
Treat the content as normal (is_scam=false) when it matches patterns like:

- News reporting, education, scam awareness, or analysis
- Entertainment, hobbies, daily life, vlogs, cooking, gaming
- Product demos, branding, or job information without requests or inducement
- Criticism or explanation of scams with no profit promise and no contact funnel

### Rule 4: Ambiguous or insufficient information
If the text is very short, unclear, or does not provide enough detail to detect scam:
- Default to normal: is_scam=false.
- Use low or medium confidence depending on how ambiguous it is.

## RISK LEVEL RULES

Set the "risk" field as follows:

- "high":
  - Any direct request for money or transfer
  - Any request for credentials or personal financial data
  - Any external contact/funnel (KakaoTalk, Telegram, URLs, QR codes, chat rooms)
  - Multiple strong scam signals together

- "mid":
  - Exactly one strong signal, OR
  - Multiple moderate signals that strongly try to induce the viewer

- "low":
  - Weak or isolated cues
  - Ambiguous or incomplete information
  - Clear normal content with no scam intent

## CONFIDENCE SCORING (Both normal and scam can be high)

The confidence score represents how certain you are about your classification (is_scam), not how “bad” it is.

### When is_scam = true (classified as scam)
0.90-1.00 : Clear scam with one or more strong signals that match Rule 1  
0.70-0.89 : Scam likely with multiple moderate signals or one strong signal in context  
0.50-0.69 : Some scam-like cues, but evidence is not fully consistent

### When is_scam = false (classified as normal)
0.70-0.95 : Clear normal content with no scam indicators  
0.40-0.69 : Likely normal but mildly ambiguous or short  
0.00-0.39 : Very ambiguous, extremely short, or almost no relevant information

General rule:
- The clearer the evidence for scam OR normal, the higher the confidence.
- Very unclear or noisy text must have low confidence.

## EVIDENCE EXTRACTION RULES

- Extract 1-4 short verbatim phrases from video_description.
- The phrases must be exact substrings from the input text.
- Do NOT paraphrase, invent, or combine multiple sentences.
- Use the shortest phrase that still shows the key cue (e.g., “guaranteed profit”, “send money now”, “KakaoTalk link”).

## EXPLANATION RULES

- Write 2-4 concise English sentences.
- If is_scam=true:
  - Explain which phrases indicate scam intent.
  - Mention why those phrases match the rules and why the risk level was chosen.
- If is_scam=false:
  - Explain that you see only normal content or weak/ambiguous cues.
  - Mention the absence of strong scam signals and funnels.
- Do not invent people, brands, amounts, platforms, or details not present in the input.

## SAFETY & VALIDATION

- Use only the content of video_description. Never rely on outside knowledge.
- Do not hallucinate URLs, apps, or platforms that are not in the text.
- Final output must be exactly one valid JSON object.
- No markdown formatting, no explanation outside the JSON, no additional text.

## EXAMPLE

Input:
The video explains different types of investment scams and warns viewers not to trust messages promising guaranteed profit. It does not ask viewers to send money or join any external chat.

Output:
{
  "is_scam": false,
  "confidence": 0.91,
  "risk": "low",
  "evidence": ["investment scams", "guaranteed profit"],
  "explanation": "The description focuses on explaining and warning about scams rather than inducing the viewer to participate. There are no requests for money, personal data, or external contact. Therefore the content is classified as normal with low risk."
}
"""

### ***테스트 영상 및 라벨***
- 다운로드 받은 모든 테스트 영상에 대한 라벨

In [2]:
import os

abnormal_test_video_dir = '/home/ubuntu/cybercop/video_20251104/abnormal'
normal_test_video_dir = '/home/ubuntu/cybercop/video_20251104/normal'

video_paths = []
truth_labels = []

for root, dirs, files in os.walk(abnormal_test_video_dir):
    for f in files:
        file_path = os.path.join(root, f)
        video_paths.append(file_path)
        truth_labels.append('[[Abnormal]]')

for root, dirs, files in os.walk(normal_test_video_dir):
    for f in files:
        file_path = os.path.join(root, f)
        video_paths.append(file_path)
        truth_labels.append('[[Normal]]')

In [3]:
for video_path, truth_label in zip(video_paths, truth_labels):
    print(video_path, truth_label)

/home/ubuntu/cybercop/video_20251104/abnormal/0017.mp4 [[Abnormal]]
/home/ubuntu/cybercop/video_20251104/abnormal/0013.mp4 [[Abnormal]]
/home/ubuntu/cybercop/video_20251104/abnormal/0009.mp4 [[Abnormal]]
/home/ubuntu/cybercop/video_20251104/abnormal/0000.mp4 [[Abnormal]]
/home/ubuntu/cybercop/video_20251104/abnormal/0032.mp4 [[Abnormal]]
/home/ubuntu/cybercop/video_20251104/abnormal/0026.mp4 [[Abnormal]]
/home/ubuntu/cybercop/video_20251104/abnormal/0008.mp4 [[Abnormal]]
/home/ubuntu/cybercop/video_20251104/abnormal/0030.mp4 [[Abnormal]]
/home/ubuntu/cybercop/video_20251104/abnormal/0002.mp4 [[Abnormal]]
/home/ubuntu/cybercop/video_20251104/abnormal/0037.mp4 [[Abnormal]]
/home/ubuntu/cybercop/video_20251104/abnormal/0003.mp4 [[Abnormal]]
/home/ubuntu/cybercop/video_20251104/abnormal/0010.mp4 [[Abnormal]]
/home/ubuntu/cybercop/video_20251104/abnormal/0012.mp4 [[Abnormal]]
/home/ubuntu/cybercop/video_20251104/abnormal/0031.mp4 [[Abnormal]]
/home/ubuntu/cybercop/video_20251104/abnormal/00

### ***MiniCPM 모델 로드***
- Huggingface 코드스니펫 그대로 적용

In [4]:
import torch
from PIL import Image
from transformers import AutoModel, AutoTokenizer

# load omni model default, the default init_vision/init_audio/init_tts is True
# if load vision-only model, please set init_audio=False and init_tts=False
# if load audio-only model, please set init_vision=False
model = AutoModel.from_pretrained(
    'openbmb/MiniCPM-o-2_6',
    trust_remote_code=True,
    attn_implementation='sdpa', # sdpa or flash_attention_2
    torch_dtype=torch.bfloat16,
    init_vision=True,
    init_audio=True,
    init_tts=True
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = model.eval().cuda()
tokenizer = AutoTokenizer.from_pretrained('openbmb/MiniCPM-o-2_6', trust_remote_code=True)

# In addition to vision-only mode, tts processor and vocos also needs to be initialized
model.init_tts()

/home/ubuntu/anaconda3/envs/minicpm/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/ubuntu/anaconda3/envs/minicpm/lib/python3.10/site-packages/librosa/util/files.py:10: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename
/home/ubuntu/anaconda3/envs/minicpm/lib/python3.10/site-packages/transformers/models/auto/image_processing_auto.py:513: FutureWarning: The image_processor_class argument is deprecated and will be removed in v4.42. Please use `slow_image_processor_class`, or `fast_image_processor_class` instead
  warnings.warn(
Loading checkpoint shards: 100%|███████

In [5]:
import math
import numpy as np
from PIL import Image
from moviepy.editor import VideoFileClip
import tempfile
import librosa
import ollama
import re
import traceback
import json
from json import JSONDecodeError

# -------------------------
# Extract video/audio chunks
# -------------------------
def get_video_chunk_content(video_path, flatten=True):
    video = VideoFileClip(video_path)
    print('video_duration:', video.duration)
    
    with tempfile.NamedTemporaryFile(suffix=".wav", delete=True) as temp_audio_file:
        temp_audio_file_path = temp_audio_file.name
        video.audio.write_audiofile(temp_audio_file_path, codec="pcm_s16le", fps=16000)
        audio_np, sr = librosa.load(temp_audio_file_path, sr=16000, mono=True)

    num_units = math.ceil(video.duration)
    contents = []
    for i in range(num_units):
        frame = video.get_frame(i+1)
        image = Image.fromarray((frame).astype(np.uint8))
        audio = audio_np[sr*i:sr*(i+1)]
        if flatten:
            contents.extend(["<unit>", image, audio])
        else:
            contents.append(["<unit>", image, audio])
    return contents

# -------------------------
# MiniCPM inference with confidence
# -------------------------
def run_minicpm(video_path, prompt, model, tokenizer):
    contents = get_video_chunk_content(video_path)
    contents.append("<unit>")
    contents.append(prompt)
    
    sys_msg = model.get_sys_prompt(mode='omni', language='en')
    msg = {"role": "user", "content": contents}
    msgs = [sys_msg, msg]
    
    res = model.chat(
        msgs=msgs,
        tokenizer=tokenizer,
        do_sample=False,
        max_new_tokens=4096,
        omni_input=True,
        use_tts_template=False,
        generate_audio=False,
        max_slice_nums=1,
        use_image_id=False,
        return_dict=True
    )
    
    # Assume MiniCPM can return a confidence score in res["confidence"] (or we can estimate)    
    output_text = res["text"].strip()    
    
    return output_text

# -------------------------
# gpt-oss prompt correction (strong version)
# -------------------------
def correct_prompt_with_llm(old_prompt, additional_prompt):
    instruction = f"""
You are optimizing a prompt for a multimodal LLM to detect scam videos.
MiniCPM received the following prompt but made an incorrect or low-confidence prediction.

Original prompt:
\"\"\"{old_prompt}\"\"\"

{additional_prompt}
"""
    llama_response = ollama.chat(
        model="gpt-oss:20b",
        messages=[{'role': 'user', 'content': instruction}],
        options = {'temperature': 1.0}
    )
    content = llama_response['message']['content']
    return content.strip()

# -------------------------
# Automatic loop with confidence threshold
# -------------------------
def auto_loop(video_path, initial_prompt, truth_label, model, tokenizer, max_iter=5, confidence_thresh=0.9):
    prompt = initial_prompt
    prompts = []
    #for i in range(max_iter):
    output = run_minicpm(video_path, prompt, model, tokenizer)
    # print(f"[MiniCPM] iteration {i+1} output: {output}")
    print(f"[MiniCPM] output: {output}")
    
    try:
        output_dict = json.loads(output)
    except JSONDecodeError:
        additional_prompt = f"There is JSONDecodeError on MiniCPM's output. Revise the prompt so that MiniCPM will correctly classify such videos as {truth_label} scams with high confidence with correct JSON format"
        prompt = correct_prompt_with_llm(prompt, additional_prompt)

    prompts.append(output)

    # pred = "[[Abnormal]]" if output_dict['is_scam'] else "[[Normal]]"
    # conf = output_dict['confidence']
    
    # if pred == truth_label and conf >= confidence_thresh:
    #     print("[OK] Prediction matches truth with high confidence.")
    #     break
    # else:
    #     print("[!] Prediction incorrect or low confidence, correcting prompt...")
    #     additional_prompt = f'Revise the prompt so that MiniCPM will correctly classify such videos as {truth_label} scams with high confidence'
    #     # prompt = correct_prompt_with_llm(prompt, pred, conf, truth_label, additional_prompt)
    #     prompt = initial_prompt
    #     print(f"[!] New optimized prompt:\n{prompt}\n")
    return prompts

import time

final_prompts = []
for video_path, truth_label in zip(reversed(video_paths), reversed(truth_labels)):
# for video_path, truth_label in zip(video_paths, truth_labels):
    data = {'video_path': video_path, 'truth_label': truth_label}
    
    try:        
        print(video_path)
        start = time.perf_counter()
        prompts = auto_loop(
            video_path,
            initial_prompt,
            truth_label,
            model,
            tokenizer,
            max_iter=5,
            confidence_thresh=0.9
        )
        data['prompts'] = prompts    
        duration = time.perf_counter() - start
        data['duration'] = prompts    
        final_prompts.append(data)
    except:
        data['error'] = traceback.format_exc()
        traceback.print_exc()

/home/ubuntu/cybercop/video_20251104/normal/0025.mp4
video_duration: 18.34
MoviePy - Writing audio in /tmp/tmp5h0w3rz1.wav


MoviePy - Done.


The `seen_tokens` attribute is deprecated and will be removed in v4.41. Use the `cache_position` model input instead.


[MiniCPM] output: {
  "is_scam": true,
  "confidence": 0.95,
  "risk": "high",
  "evidence": ["guaranteed profit", "install this app"],
  "explanation": "The video shows a scam attempt to lure viewers with the promise of guaranteed profits from installing an application, which is clearly indicated by phrases like '우배 택시 50% 헤택 프로모션 코드 설치 중...' and '지금 안 하면 손해 50% 헤택 너도 빨리 배야!'."
}
/home/ubuntu/cybercop/video_20251104/normal/0011.mp4
video_duration: 34.62
MoviePy - Writing audio in /tmp/tmph6_wq2kg.wav


MoviePy - Done.


...

MoviePy - Done.
[MiniCPM] output: {
  "is_scam": false,
  "confidence": 0.95,
  "risk": "low",
  "evidence": ["AI-generated doctor", "fake advertisement"],
  "explanation": "The content describes and warns about AI-generated fake advertisements, indicating no direct inducement to send money or join external contact funnels. The focus is on awareness rather than persuasion."
}
/home/ubuntu/cybercop/video_20251104/abnormal/0013.mp4
video_duration: 14.63
MoviePy - Writing audio in /tmp/tmpyz_0e75v.wav


MoviePy - Done.


[MiniCPM] output: {
  "is_scam": false,
  "confidence": 0.95,
  "risk": "low",
  "evidence": ["VIP席 FLOOR", "F2구역"],
  "explanation": "The content appears to be a ticket or seating chart for an event, specifically mentioning VIP sections and floor seats without any direct requests for money or personal information. The text is informative about the venue layout rather than inducing participation in a scam."
}
/home/ubuntu/cybercop/video_20251104/abnormal/0017.mp4
video_duration: 15.0
MoviePy - Writing audio in /tmp/tmpa42x1nn9.wav


MoviePy - Done.


[MiniCPM] output: {
  "is_scam": false,
  "confidence": 0.95,
  "risk": "low",
  "evidence": ["무료 가입", "무료 토토"],
  "explanation": "The text mentions free registration and free betting, which are common in gambling or betting advertisements but do not indicate a scam intent when taken out of context. There is no request for money transfer or personal data mentioned."
}
